# T2 WP-Frag current-evidence rerun — `ablate_no_mass_prior`

**Goal.** Reproduce current-main (`g3_5_7` mirror) evidence for this one model so the
paper Results section (chapter section 8) can be cited from a single, machine-reproducible
suite instead of catalog-era cloud numbers contaminated by environment drift.

**Matrix (this notebook).** `ablate_no_mass_prior` x {clean, iid_noisy_ic@nominal_train, v4_lite@nominal_train} x seeds {42,43,44,45,46} = 15 training runs, then every trained
suite is evaluated across {clean, nominal_eval, degraded_eval, heading_biased_eval}.

**Parallelism.** Run the four `t2_wpfrag_*` notebooks (one per model:
`phnode_full`, `phnode_qforce`, `ablate_no_lift`, `ablate_no_mass_prior`) in separate
Colab sessions. They use distinct `RUN_TAG`s and do not collide.

**Why these knobs differ from phase1a:** (1) `IID_EVAL_PROFILES` is the full 4-profile
robustness set so clean-vs-noisy matched comparison is available; (2) a provenance cell
writes per-run `_audit_meta/` (the trainer does not). noise schedule defaults
(warmup=20, ramp=80, mix_ratio=0.5) match the repo `nominal_train` contract.

**After all four finish:** sync `checkpoints/` back, then locally rebuild the catalog
(`scripts/build_oc_data_catalog.py`) and export the section-8 current-evidence tables.

> Watch list while running: `phnode_full` seeds 42/46, `ablate_no_lift` seeds 43/44 —
> these were catalog-era anomalies. The anomaly-scan cell flags any recurrence.

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA: {torch.version.cuda}")
print(f"cuDNN: {torch.backends.cudnn.version()}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path(os.environ.get(
    "AUV_PROJECT_DIR",
    "/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7",
))
assert PROJECT_DIR.exists(), f"Project directory not found: {PROJECT_DIR}"
%cd $PROJECT_DIR

In [ ]:
%pip install -q torchdiffeq pandas

## 0. Configuration

In [ ]:
import os

# Runtime
os.environ["PYTHON_BIN"] = "python"
os.environ["DEVICE"] = "cuda"
os.environ["LOCAL_PROXY_ROOT"] = "/content/_proxy_suites"

# ---- T2 identity (one model per notebook) ----
os.environ["RUN_TAG"] = "t2_wpfrag_ablate_no_mass_prior"
os.environ["DATASET"] = "data/auv_oc_traj1000_blk150_s23_d0be9434.pkl"
os.environ["NOISE_REFERENCE"] = "remus100_dr"
os.environ["PHASE1A_LOG_DIR"] = str(PROJECT_DIR / "checkpoints" / "phase1a_logs" / os.environ["RUN_TAG"])
os.environ["PHASE1A_METADATA_DIR"] = str(PROJECT_DIR / "checkpoints" / f"phase1a_metadata_{os.environ['RUN_TAG']}")

# ---- T2 matrix: single model x 3 protocols x 5 seeds ----
os.environ["PHASE1A_MODELS"] = "ablate_no_mass_prior"
os.environ["SMOKE1_MODELS"] = "ablate_no_mass_prior"
os.environ["SMOKE_SEEDS"] = "42"                 # cheap protocol-correctness gate (1 seed)
os.environ["DECISION_SEEDS"] = "42 43 44 45 46"  # full current-evidence seed set

# ---- Evaluation contract ----
os.environ["SMOKE_EVAL_NUM_TRAJ_PER_SCENARIO"] = "6"
os.environ["DECISION_EVAL_NUM_TRAJ_PER_SCENARIO"] = "30"
os.environ["EVAL_TIMES"] = "10 30 60"
os.environ["EVAL_SCENARIOS"] = "PRBS CHIRP OU"
os.environ["EVAL_BASE_SEED"] = "42"
os.environ["EVAL_NOISE_SEED"] = "2024"
os.environ["EVAL_PROGRESS_EVERY"] = "5"
os.environ["EVAL_NUM_DIAGNOSTIC_PLOTS"] = "6"
# Full robustness set: evaluates clean/iid/v4lite suites across all 4 profiles (caveat B).
os.environ["IID_EVAL_PROFILES"] = "clean nominal_eval degraded_eval heading_biased_eval"
os.environ["V4_EVAL_PROFILES"] = "nominal_eval"

# ---- Audit gate ----
os.environ["STRICT_ZERO_NOISE_AUDIT"] = "1"
os.environ["SOFT_MIN_EPOCH_SCALE"] = "0.05"

print("RUN_TAG       =", os.environ["RUN_TAG"])
print("PHASE1A_MODELS=", os.environ["PHASE1A_MODELS"])
print("DECISION_SEEDS=", os.environ["DECISION_SEEDS"])
print("IID_EVAL_PROFILES=", os.environ["IID_EVAL_PROFILES"])

## 1. Preflight
Confirms all target suite/proxy dirs are absent and saves suite-level run config + environment metadata. If it fails, change `RUN_TAG` or remove the target dirs.

In [ ]:
os.environ["MODE"] = "preflight"
!bash scripts/run_phase1a_oc_v4lite.sh

## 2. Smoke gate (seed 42, all 3 protocols)
Cheap protocol-correctness check (especially the `v4_lite` trajectory-consistent IC path and the strict zero-noise audit for `clean`). Important for `phnode_qforce`, which was not in the phase1a smoke set. Smoke results are flow-validation only — never cited.

In [ ]:
os.environ["MODE"] = "smoke1_train"
!bash scripts/run_phase1a_oc_v4lite.sh

In [ ]:
os.environ["MODE"] = "smoke1_eval"
!bash scripts/run_phase1a_oc_v4lite.sh

## 3. Decision train (3 protocols x 5 seeds)
The 15 evidence-bearing training runs for this model.

In [ ]:
os.environ["MODE"] = "decision_train"
!bash scripts/run_phase1a_oc_v4lite.sh

## 4. Decision rollout eval (each trained suite x 4 robustness profiles)
Evaluates the clean / iid / v4lite trained suites across `clean nominal_eval degraded_eval heading_biased_eval` (iid eval protocol) plus the `v4_lite` eval at `nominal_eval`.

In [ ]:
os.environ["MODE"] = "decision_eval"
!bash scripts/run_phase1a_oc_v4lite.sh

## 5. Provenance injection (per-run `_audit_meta/`)
Records git HEAD + environment fingerprint into every decision run dir so these runs are distinguishable from catalog-era drift when the catalog is rebuilt.

In [ ]:
# Inject per-run provenance. The training flow does NOT write these automatically,
# so for paper-grade evidence we record them here (see docs/provenance_audit_phnode_full_clean.md sec 5.2).
import os, sys, subprocess, datetime
from pathlib import Path
import torch

RUN_TAG = os.environ["RUN_TAG"]
ckpt = PROJECT_DIR / "checkpoints"
suites = [ckpt / f"sweep_oc_phase1a_decision_{p}_{RUN_TAG}" for p in ("clean", "iid", "v4lite")]

def _sh(args):
    try:
        return subprocess.run(args, cwd=PROJECT_DIR, capture_output=True, text=True).stdout.strip()
    except Exception as exc:  # noqa: BLE001
        return f"<unavailable: {exc}>"

head = _sh(["git", "rev-parse", "HEAD"])
diffstat = _sh(["git", "diff", "HEAD", "--stat"])
env_txt = "\n".join([
    f"python={sys.version.split()[0]}",
    f"torch={torch.__version__}",
    f"cuda={torch.version.cuda}",
    f"cudnn={torch.backends.cudnn.version()}",
    f"gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}",
    f"captured_at={datetime.datetime.now().isoformat()}",
])

n = 0
for suite in suites:
    if not suite.exists():
        continue
    for cfg in suite.rglob("config.json"):
        run_dir = cfg.parent
        meta = run_dir / "_audit_meta"
        meta.mkdir(exist_ok=True)
        (meta / "code_revision.txt").write_text(f"git_head={head}\n\n{diffstat}\n")
        (meta / "environment.txt").write_text(env_txt + "\n")
        n += 1
print(f"provenance written for {n} run dirs")
print("git_head =", head)

## 6. Anomaly scan
Flags any run with the `no successful training batches` catastrophic-gradient signature (the seed46 failure mode). A flagged seed must NOT be cited — rerun or annotate.

In [ ]:
# Quick anomaly scan: catch seed46/seed43-style catastrophic training before trusting numbers.
# Flags any run whose training.log contains the "no successful training batches" failure mode.
import os, re
from pathlib import Path

RUN_TAG = os.environ["RUN_TAG"]
ckpt = PROJECT_DIR / "checkpoints"
suites = [ckpt / f"sweep_oc_phase1a_decision_{p}_{RUN_TAG}" for p in ("clean", "iid", "v4lite")]

print(f"{'suite':<48}{'run':<32}{'best_epoch':>10}{'best_loss':>14}{'nbad':>6}  flag")
for suite in suites:
    if not suite.exists():
        continue
    for cfg in sorted(suite.rglob("config.json")):
        run_dir = cfg.parent
        log = run_dir / "training.log"
        text = log.read_text() if log.exists() else ""
        nbad = text.count("no successful training batches")
        best_epoch, best_loss = "?", "?"
        m = re.findall(r"[Bb]est.*?epoch[^0-9]*([0-9]+).*?([0-9]+\.[0-9eE+-]+)", text)
        if m:
            best_epoch, best_loss = m[-1]
        flag = "  <-- CHECK (possible artifact)" if nbad > 0 else ""
        print(f"{suite.name[:46]:<48}{run_dir.name[:30]:<32}{str(best_epoch):>10}{str(best_loss):>14}{nbad:>6}{flag}")
print("\nIf any run is flagged, treat that seed like seed46/seed43: do NOT cite it; rerun or annotate.")

## 7. Next step (after all 4 notebooks finish)
1. Ensure `checkpoints/sweep_oc_phase1a_decision_{clean,iid,v4lite}_t2_wpfrag_*` are synced back to the repo.
2. Locally: `conda run -n mytorch1 python scripts/build_oc_data_catalog.py` to ingest the new runs.
3. Export the section-8 current-evidence tables from the rebuilt canonical views.
4. Re-verify the `ablate_no_lift` noisy-degradation claim is not driven by a seed44 artifact.